# CENTROIDFOLD
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
!conda install -c bioconda centroid_rna_package

In [ ]:
method_name = "CentroidFold"
base = Path.cwd()

result = subprocess.run(['centroid_fold', '--version'], capture_output=True, text=True)

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(name: str, seq: str):
    tmp_fasta = f'CentroidFold_tmp_{name}.fasta'
    out_file_name = f'CentroidFold_clean_tmp_{name}.dot'
    with open(tmp_fasta, 'w') as ofile:
        ofile.write(f'>{name}\n{seq}\n')
    cmd = f"centroid_fold {tmp_fasta} | cut -d' ' -f1 > {out_file_name}"
    subprocess.run(cmd, shell=True, check=True)
    os.remove(tmp_fasta)
    with open(out_file_name, 'r') as f:
        lines = f.readlines()
        structure = lines[-1].strip()
    os.remove(out_file_name)
    return structure

In [ ]:
os.makedirs('../prediction', exist_ok=True)
out_fasta_name = f'../prediction/{method_name}.fasta'
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<25}\t{'len':<5}\t{'time':<8}")
print("-" * 70)

with open(out_fasta_name, 'w') as output_file:
    for i, vid in enumerate(virus_ids):
        start_time = time.time()
        seq = viruses.loc[vid]['sequence']
        print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<25}\t{len(seq):<5}\t", end='', flush=True)
        
        structure = run_folding(vid, seq)
        elapsed_time = time.time() - start_time
        
        output_file.write(f'>{vid}\n')
        output_file.write(f'{seq}\n')
        output_file.write(f'{structure}\n')
        
        print(f"{elapsed_time:6.1f}s")
        
print(f"\nProcessing complete. Results saved to {out_fasta_name}.fasta")